# Занятие 10. Семинар: слои нейронной сети на numpy

Пишем руками четыре модуля сети — полносвязный слой и три функции активации, — проверяем
их численным градиентом, собираем из них сеть из лекции с настраиваемыми числом слоев,
активацией и инициализацией и сравниваем варианты на рукописных цифрах MNIST. Выводы
градиентов даны ниже; пустые ячейки дописываем на паре.

У каждого модуля три метода:

- `forward(X)` — выход по входу; то, что понадобится в обратном проходе, модуль запоминает;
- `backward(grad_output)` — получает $G = \frac{\partial L}{\partial Y}$, градиент потери по своему выходу, считает градиенты
  по своим параметрам и возвращает $\frac{\partial L}{\partial X}$ — его получит предыдущий модуль;
- `parameters()` — пары (параметр, градиент) для оптимизатора; у функций активации их нет.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml

from sem10_utils import plot_histories, show_digits, show_table

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42

In [ ]:
mnist_X, mnist_y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="liac-arff")
mnist_X = (mnist_X / 255.0).astype(np.float64)       # яркость пикселя от 0 до 1
mnist_y = mnist_y.astype(int)

# стандартное деление MNIST: первые 60 000 — обучение, последние 10 000 — тестовый сет
X_train, y_train = mnist_X[:50000], mnist_y[:50000]
X_val, y_val = mnist_X[50000:60000], mnist_y[50000:60000]
X_test, y_test = mnist_X[60000:], mnist_y[60000:]

print("обучение:", X_train.shape, "валидация:", X_val.shape, "тест:", X_test.shape)

In [ ]:
show_digits(X_train[:20], y_train[:20], title="первые 20 картинок обучающей выборки: над каждой — ее класс")

## 0. Шпаргалка по матричному дифференцированию

Развернутый вариант с выводами всех формул — в файле `matrix_calculus.pdf` рядом с ноутбуком.
Здесь — то, что понадобится в ближайший час.

**Что считается производной.** Градиент по аргументу имеет ту же форму, что аргумент:
по вектору $x \in \mathbb{R}^n$ — вектор $\nabla f \in \mathbb{R}^n$, по матрице $W \in \mathbb{R}^{n \times m}$ — матрица $\frac{\partial f}{\partial W} \in \mathbb{R}^{n \times m}$
с элементами $\frac{\partial f}{\partial W_{ij}}$. Для отображения $g\colon \mathbb{R}^n \to \mathbb{R}^p$ производная — матрица Якоби
$J_g \in \mathbb{R}^{p \times n}$, $(J_g)_{kj} = \frac{\partial g_k}{\partial x_j}$.

**Цепное правило.** Для $h(x) = f(g(x))$ со скалярной $f$:

$$\nabla h(x) = J_g(x)^\top\, \nabla f(g(x)), \qquad (n \times 1) = (n \times p)(p \times 1).$$

Обратный проход — это же правило, примененное справа налево: слой получает градиент по своему
выходу и умножает его на транспонированную матрицу Якоби. Саму матрицу не строят — для каждого
слоя произведение записывают напрямую.

**Линеаризация.** Градиент — то, что стоит в скалярном произведении с приращением:
$f(x + \Delta) = f(x) + \langle \nabla f, \Delta \rangle + o(\|\Delta\|)$; для матриц $\langle A, B \rangle = \operatorname{tr}(A^\top B) = \sum_{ij} A_{ij} B_{ij}$.
Удобный способ вывести формулу: раскрыть $f(x + \Delta)$ и собрать все линейное по $\Delta$.

**Обозначение для слоев.** Слой с входом $X$ и выходом $Y$ стоит внутри сети, и потеря $L$ зависит
от его выхода. Градиент потери по выходу слоя обозначим $G = \frac{\partial L}{\partial Y}$ — матрица той же формы,
что $Y$; именно его слой получает аргументом `backward`, а вернуть должен $\frac{\partial L}{\partial X}$.

**Таблица.** $a$, $A$ — постоянные.

| функция | градиент | | функция | градиент |
|:--|:--|:--|:--|:--|
| $a^\top x$ | $a$ | | $x^\top A x$ | $(A + A^\top)\,x$ |
| $\|x\|^2 = x^\top x$ | $2x$ | | $\tfrac12 \|Ax - b\|^2$ | $A^\top (Ax - b)$ |
| $\varphi(a^\top x)$ | $\varphi'(a^\top x)\, a$ | | $\operatorname{tr}(A^\top X) = \langle A, X \rangle$ | $A$ |
| $Y = XW + b$ | $\frac{\partial L}{\partial W} = X^\top G$, $\frac{\partial L}{\partial b} = \sum_i G_{i\cdot}$, $\frac{\partial L}{\partial X} = G W^\top$ | | $Y = \varphi(X)$ поэлементно | $\frac{\partial L}{\partial X} = G \odot \varphi'(X)$ |

**Как проверять себя:** размеры должны стыковаться; при $n = 1$ формула должна стать школьной;
численный градиент — центральная разность с $\varepsilon = 10^{-5}$ — должен совпасть с относительной
ошибкой меньше $10^{-6}$.

## 1. Полносвязный слой

Пачка из $n$ объектов — матрица $X \in \mathbb{R}^{n \times n_{in}}$, выход $Y = XW + b$, то есть $Y_{ij} = \sum_k X_{ik} W_{kj} + b_j$.
В `backward` приходит $G = \frac{\partial L}{\partial Y}$. Потеря зависит от $W_{kj}$ через все $Y_{ij}$ столбца $j$,
и $\frac{\partial Y_{ij}}{\partial W_{kj}} = X_{ik}$, поэтому по цепному правилу

$$\frac{\partial L}{\partial W_{kj}} = \sum_i G_{ij} X_{ik} = (X^\top G)_{kj} \quad\Rightarrow\quad \frac{\partial L}{\partial W} = X^\top G.$$

Сдвиг $b_j$ входит в каждую строку с коэффициентом 1, а вход $X_{ik}$ — во все $Y_{ij}$ строки $i$
с коэффициентом $W_{kj}$:

$$\frac{\partial L}{\partial b_j} = \sum_i G_{ij}, \qquad \frac{\partial L}{\partial X_{ik}} = \sum_j G_{ij} W_{kj} = (G W^\top)_{ik} \quad\Rightarrow\quad \frac{\partial L}{\partial X} = G W^\top.$$

Проверка размеров: $X^\top G$ — $(n_{in} \times n)(n \times n_{out})$, как $W$; $G W^\top$ — $(n \times n_{out})(n_{out} \times n_{in})$, как $X$.
Начальные веса $W_{kj} \sim \mathcal{N}(0, \frac{2}{n_{in}})$ — инициализация He, сдвиги — нули.

In [ ]:
class Linear:
    """Полносвязный слой: Y = X W + b."""

    # W: (n_in, n_out) ~ N(0, 2 / n_in) через rng.normal; b: (n_out,) нули; dW, db — нули той же формы
    def __init__(self, n_in: int, n_out: int, rng: np.random.Generator):
        ...

    # X: (n, n_in) -> (n, n_out); X запомнить для backward
    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    # grad_output = dL/dY: (n, n_out); записывает self.dW, self.db; возвращает dL/dX: (n, n_in)
    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    def parameters(self) -> list:
        pairs = [(self.W, self.dW), (self.b, self.db)]
        return pairs

## 2. Функции активации

Функция активации применяется к каждому элементу отдельно, $Y_{ij} = \varphi(X_{ij})$, поэтому
по цепному правилу градиент по входу — градиент по выходу, умноженный поэлементно на производную:

$$\frac{\partial L}{\partial X_{ij}} = G_{ij}\, \varphi'(X_{ij}).$$

**Сигмоида** $\sigma(z) = \frac{1}{1 + e^{-z}}$. Производная:

$$\sigma'(z) = \frac{e^{-z}}{(1 + e^{-z})^2} = \frac{1}{1 + e^{-z}}\cdot\frac{e^{-z}}{1 + e^{-z}} = \sigma(z)\,\big(1 - \sigma(z)\big),$$

поэтому в `backward` нужен только выход прямого прохода. При $z = -1000$ выражение $e^{-z}$
переполняется; устойчивая запись — $\sigma(z) = \frac{1}{2}\big(1 + \tanh\frac{z}{2}\big)$.

**ReLU** $\max(z, 0)$: производная $[z > 0]$ — 1 при положительном входе и 0 иначе.

**Leaky ReLU** $\max(z, \alpha z)$ при $0 < \alpha < 1$: производная 1 при $z > 0$ и $\alpha$ иначе.

In [ ]:
# у всех трех: forward(X) -> та же форма, backward(grad_output) -> градиент по входу той же формы
class Sigmoid:
    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    def parameters(self) -> list:
        return []


class ReLU:
    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    def parameters(self) -> list:
        return []


class LeakyReLU:
    def __init__(self, alpha: float = 0.01):
        self.alpha = alpha

    def forward(self, X: np.ndarray) -> np.ndarray:
        return ...

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return ...

    def parameters(self) -> list:
        return []

## 3. Проверка градиентов

Численная производная — центральная разность $\frac{f(x + \varepsilon) - f(x - \varepsilon)}{2\varepsilon}$ с ошибкой порядка $\varepsilon^2$.
Для проверки модуля берем случайный вход $X$ и случайную матрицу $R$ той же формы, что выход,
и потерю $L = \sum_{ij} Y_{ij} R_{ij}$: ее градиент по выходу — ровно $R$. Сравниваем `backward(R)`
с численным градиентом по входу и по параметрам; относительная ошибка меньше $10^{-6}$ — модуль верный.

In [ ]:
def numeric_gradient(f, x: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """Центральная разность по каждой координате x: (f(x + eps·e_i) − f(x − eps·e_i)) / (2 eps)."""
    gradient = np.zeros_like(x)
    for index in np.ndindex(x.shape):
        original = x[index]

        x[index] = original + eps
        loss_plus = f()
        x[index] = original - eps
        loss_minus = f()
        x[index] = original

        gradient[index] = (loss_plus - loss_minus) / (2 * eps)
    return gradient


def relative_error(a: np.ndarray, b: np.ndarray) -> float:
    error = float(np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b))))
    return error


def check_module(module, X: np.ndarray, rng: np.random.Generator) -> dict:
    """Относительные ошибки backward по входу и по каждому параметру модуля."""
    R = rng.normal(size=module.forward(X).shape)
    loss = lambda: float(np.sum(module.forward(X) * R))

    module.forward(X)
    grad_input = module.backward(R)
    errors = {"по входу": relative_error(grad_input, numeric_gradient(loss, X))}

    analytic = [gradient.copy() for _, gradient in module.parameters()]
    for number, ((parameter, _), gradient) in enumerate(zip(module.parameters(), analytic)):
        errors[f"по параметру {number}"] = relative_error(gradient, numeric_gradient(loss, parameter))
    return errors


check_rng = np.random.default_rng(0)
X_check = check_rng.normal(size=(5, 4))
X_check[np.abs(X_check) < 0.05] = 0.3          # подальше от излома ReLU в нуле
for name, module in [("Linear", Linear(4, 3, check_rng)), ("Sigmoid", Sigmoid()), ("ReLU", ReLU()),
                     ("LeakyReLU", LeakyReLU(0.1))]:
    errors = check_module(module, X_check.copy(), check_rng)
    print(f"{name:10s}", {key: f"{value:.1e}" for key, value in errors.items()})

## 4. Сеть из нескольких слоев

Последний модуль — софтмакс с кросс-энтропией. Его градиент по оценкам
$\frac{\partial L}{\partial z_{ik}} = \frac{1}{n}\big(p_{ik} - [k = y_i]\big)$ — тот же $p - y$, что в логистической регрессии
и в бустинге для классификации. `Sequential` вызывает `forward` модулей слева направо
и `backward` справа налево; `SGD` делает шаг $\theta \leftarrow \theta - \eta\,\frac{\partial L}{\partial \theta}$; `train` проходит по выборке
мини-батчами по 64 картинки.

In [ ]:
class SoftmaxCrossEntropy:
    """Средняя кросс-энтропия софтмакса оценок по объектам."""

    def forward(self, logits: np.ndarray, y: np.ndarray) -> float:
        shifted = logits - logits.max(axis=1, keepdims=True)        # устойчивость: exp не переполняется
        exp_scores = np.exp(shifted)
        self.probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        self.y = y

        n = len(y)
        true_class_probability = self.probabilities[np.arange(n), y]
        loss = float(-np.mean(np.log(true_class_probability + 1e-12)))
        return loss

    def backward(self) -> np.ndarray:
        n = len(self.y)
        grad_logits = self.probabilities.copy()
        grad_logits[np.arange(n), self.y] -= 1.0
        grad_logits = grad_logits / n
        return grad_logits


class Sequential:
    """Слои по очереди: forward слева направо, backward справа налево."""

    def __init__(self, layers: list):
        self.layers = layers

    def forward(self, X: np.ndarray) -> np.ndarray:
        output = X
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        grad = grad_output
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def parameters(self) -> list:
        pairs = [pair for layer in self.layers for pair in layer.parameters()]
        return pairs


class SGD:
    """Градиентный спуск: параметр ← параметр − lr · градиент, на месте."""

    def __init__(self, lr: float):
        self.lr = lr

    def step(self, parameters: list) -> None:
        for parameter, gradient in parameters:
            parameter -= self.lr * gradient


def iterate_minibatches(X: np.ndarray, y: np.ndarray, batch_size: int, rng: np.random.Generator):
    """Перемешать объекты и выдать их пачками по batch_size; каждый объект — ровно один раз за эпоху."""
    order = rng.permutation(len(y))
    for start in range(0, len(y), batch_size):
        batch = order[start:start + batch_size]
        yield X[batch], y[batch]


def accuracy(model, X: np.ndarray, y: np.ndarray) -> float:
    predicted = model.forward(X).argmax(axis=1)
    value = float(np.mean(predicted == y))
    return value


def train(model, optimizer, X: np.ndarray, y: np.ndarray, X_val: np.ndarray, y_val: np.ndarray,
          n_epochs: int, batch_size: int, rng: np.random.Generator) -> dict:
    """Обучить сеть; по эпохам вернуть среднюю потерю на обучении и accuracy на обучении и валидации."""
    loss_function = SoftmaxCrossEntropy()
    history = {"потеря на обучении": [], "accuracy на обучении": [], "accuracy на валидации": []}

    for epoch in range(n_epochs):
        batch_losses = []
        for X_batch, y_batch in iterate_minibatches(X, y, batch_size, rng):
            logits = model.forward(X_batch)
            loss = loss_function.forward(logits, y_batch)
            grad_logits = loss_function.backward()
            model.backward(grad_logits)
            optimizer.step(model.parameters())
            batch_losses.append(loss)

        history["потеря на обучении"].append(float(np.mean(batch_losses)))
        history["accuracy на обучении"].append(accuracy(model, X, y))
        history["accuracy на валидации"].append(accuracy(model, X_val, y_val))
    return history

## 5. Сеть как в лекции: число слоев, активация, инициализация

`make_network(hidden_sizes, activation, init, rng)` собирает сеть $784 \to$ `hidden_sizes` $\to 10$:
после каждого скрытого `Linear` стоит активация из `ACTIVATIONS`, веса каждого `Linear`
задает `init_weights`. Проверять будем на сети из лекции: три скрытых слоя 256, 128, 64,
ReLU между ними, софтмакс в конце.

In [ ]:
ACTIVATIONS = {"relu": ReLU, "sigmoid": Sigmoid, "leaky_relu": LeakyReLU}


# method: "he" — N(0, 2/n_in); "xavier" — N(0, 2/(n_in + n_out)); "normal" — N(0, 1);
# "constant" — все 0.01; "zeros" — нули. Возвращает (n_in, n_out)
def init_weights(n_in: int, n_out: int, method: str, rng: np.random.Generator) -> np.ndarray:
    return ...


# n_in -> hidden_sizes[0] -> ... -> n_out; после каждого скрытого Linear — ACTIVATIONS[activation]();
# веса каждого Linear заменить на init_weights(...)
def make_network(hidden_sizes: list, activation: str, init: str, rng: np.random.Generator,
                 n_in: int = 784, n_out: int = 10) -> Sequential:
    return ...

In [ ]:
# --- проверка ---
check = make_network([256, 128, 64], "relu", "he", np.random.default_rng(0))
shapes = [layer.W.shape for layer in check.layers if isinstance(layer, Linear)]
assert shapes == [(784, 256), (256, 128), (128, 64), (64, 10)], f"формы весов {shapes}"
assert sum(isinstance(layer, ReLU) for layer in check.layers) == 3, "три ReLU между четырьмя Linear"
assert check.forward(X_train[:5]).shape == (5, 10), "выход — (n, 10)"
assert np.allclose(make_network([32], "relu", "zeros", np.random.default_rng(0)).layers[0].W, 0), "zeros — нули"
assert np.allclose(make_network([32], "relu", "constant", np.random.default_rng(0)).layers[0].W, 0.01), "constant — 0.01"
he_std = make_network([256], "relu", "he", np.random.default_rng(1)).layers[0].W.std()
assert abs(he_std - np.sqrt(2 / 784)) < 0.003, f"he: σ = {he_std:.4f}, нужно {np.sqrt(2 / 784):.4f}"
print("проверки пройдены")

**Эталон.** Сеть из лекции: `[256, 128, 64]`, ReLU, He; 5 эпох, шаг 0.1, пачки по 64.

In [ ]:
def run(hidden_sizes, activation, init, n_epochs=5, lr=0.1, seed=SEED):
    """Обучить сеть с заданной конфигурацией; вернуть историю, accuracy на тесте и время."""
    rng = np.random.default_rng(seed)
    network = make_network(hidden_sizes, activation, init, rng)
    started = time.perf_counter()
    with np.errstate(over="ignore", invalid="ignore"):
        history = train(network, SGD(lr=lr), X_train, y_train, X_val, y_val, n_epochs=n_epochs, batch_size=64, rng=rng)
    result = {
        "accuracy на валидации": history["accuracy на валидации"][-1],
        "accuracy на тесте": accuracy(network, X_test, y_test),
        "время обучения, с": time.perf_counter() - started,
        "history": history,
    }
    return result


reference = run([256, 128, 64], "relu", "he")
print(f"эталон: accuracy на тесте {reference['accuracy на тесте']:.4f}, обучение {reference['время обучения, с']:.1f} с")
plot_histories({"256 → 128 → 64, ReLU, He": reference["history"]}, accuracy_ylim=(0.9, 1.0))

## 6. Что меняет активация, инициализация и ширина

Все сравнения — в одном сетапе: те же данные, 5 эпох, шаг 0.1, пачки по 64, один генератор.
Меняем одно, остальное — как у эталона.

In [ ]:
rows = {}
histories = {}
experiments = [
    ("эталон: [256, 128, 64], ReLU, He", [256, 128, 64], "relu", "he"),
    ("сигмоида", [256, 128, 64], "sigmoid", "he"),
    ("Leaky ReLU", [256, 128, 64], "leaky_relu", "he"),
    ("Xavier", [256, 128, 64], "relu", "xavier"),
    ("N(0, 1)", [256, 128, 64], "relu", "normal"),
    ("все веса 0.01", [256, 128, 64], "relu", "constant"),
    ("уже: [64, 32, 16]", [64, 32, 16], "relu", "he"),
    ("шире: [512, 256, 128]", [512, 256, 128], "relu", "he"),
    ("мельче: [128]", [128], "relu", "he"),
]
for name, hidden_sizes, activation, init in experiments:
    result = run(hidden_sizes, activation, init)
    histories[name] = result.pop("history")
    rows[name] = result

comparison = pd.DataFrame(rows).T
show_table(comparison)

In [ ]:
plot_histories({name: histories[name] for name in ["эталон: [256, 128, 64], ReLU, He", "сигмоида", "Leaky ReLU"]},
               accuracy_ylim=(0.8, 1.0))
plot_histories({name: histories[name] for name in ["эталон: [256, 128, 64], ReLU, He", "Xavier", "N(0, 1)", "все веса 0.01"]},
               accuracy_ylim=(0.0, 1.0))
plot_histories({name: histories[name] for name in ["эталон: [256, 128, 64], ReLU, He", "уже: [64, 32, 16]", "шире: [512, 256, 128]", "мельче: [128]"]},
               accuracy_ylim=(0.9, 1.0))

**Вопросы для обсуждения.**

- Почему сигмоида на четырех слоях учится заметно медленнее ReLU? Какое наибольшее значение
  у ее производной и сколько таких множителей встречает градиент по дороге к первому слою?
- Что происходит с сетью, у которой все веса 0.01: посмотрите на accuracy и объясните через
  симметрию нейронов. Чем отличается случай $\mathcal{N}(0, 1)$?
- Шире — всегда лучше? Сеть `[512, 256, 128]` при том же шаге 0.1 дала accuracy ниже узкой.
  Посмотрите на ее потерю по эпохам: она не падает ровно, а скачет. Чем больше весов, тем
  больше сумма их градиентов и тем длиннее шаг при том же $\eta$; при каком шаге широкая сеть
  обгонит эталон? Сравните и время обучения для трех ширин.

Дальше — лабораторная: из этих модулей собирают сеть заданной архитектуры и обучают ее
на цветных картинках CIFAR-10.